In [0]:
dbutils.notebook.run(
    "/Users/aadityrathore0344@gmail.com/financial-market-intelligence/Set_UP/store_key",
    timeout_seconds=180
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load silver stocks data
print("📊 Loading silver stocks data...")
silver_stocks = spark.table("financial_market.silver.stocks")

print(f"Total records: {silver_stocks.count():,}")
print(f"Date range: {silver_stocks.agg(F.min('date')).collect()[0][0]} to {silver_stocks.agg(F.max('date')).collect()[0][0]}")
print(f"Unique tickers: {silver_stocks.select('ticker').distinct().count()}")

# Show sample
print("\n📋 Sample data:")
display(silver_stocks.orderBy(F.col("date").desc()).limit(10))

In [0]:
# ============================
# DAILY MARKET AGGREGATIONS
# ============================

print("📈 Computing daily market aggregations...\n")

# Daily aggregations across all stocks
daily_agg = silver_stocks.groupBy("date").agg(
    F.count("ticker").alias("num_stocks"),
    F.round(F.avg("close"), 2).alias("avg_close_price"),
    F.round(F.avg("volume"), 0).alias("avg_volume"),
    F.round(F.sum("volume"), 0).alias("total_volume"),
    F.round(F.avg("daily_change_pct"), 2).alias("avg_daily_change_pct"),
    F.round(F.avg("price_volatility"), 2).alias("avg_volatility"),
    F.max("high").alias("highest_price"),
    F.min("low").alias("lowest_price")
).orderBy("date")

print(f"Daily aggregations: {daily_agg.count()} trading days")
print("\n📋 Daily Market Summary:")
display(daily_agg.orderBy(F.col("date").desc()).limit(10))

In [0]:
# ============================
# TICKER-LEVEL AGGREGATIONS
# ============================

print("🎯 Computing ticker-level aggregations...\n")

# Aggregations by ticker
ticker_agg = silver_stocks.groupBy("ticker").agg(
    F.count("date").alias("trading_days"),
    F.round(F.avg("close"), 2).alias("avg_close"),
    F.round(F.stddev("close"), 2).alias("stddev_close"),
    F.round(F.min("close"), 2).alias("min_close"),
    F.round(F.max("close"), 2).alias("max_close"),
    F.round(F.avg("volume"), 0).alias("avg_volume"),
    F.round(F.avg("daily_change_pct"), 2).alias("avg_daily_change_pct"),
    F.round(F.avg("price_volatility"), 2).alias("avg_volatility"),
    F.min("date").alias("first_date"),
    F.max("date").alias("last_date")
).orderBy(F.col("avg_volume").desc())

print(f"Ticker aggregations: {ticker_agg.count()} unique tickers")
print("\n📋 Top Tickers by Average Volume:")
display(ticker_agg.limit(10))

In [0]:
# ============================
# ROLLING AVERAGES (MOVING AVERAGES)
# ============================

print("📊 Computing rolling averages (7-day, 30-day, 90-day)...\n")

# Define window specifications for rolling averages
# Window partitioned by ticker, ordered by date, with rows before current row
window_7day = Window.partitionBy("ticker").orderBy("date").rowsBetween(-6, 0)
window_30day = Window.partitionBy("ticker").orderBy("date").rowsBetween(-29, 0)
window_90day = Window.partitionBy("ticker").orderBy("date").rowsBetween(-89, 0)

# Calculate rolling averages for multiple metrics
rolling_avg_df = silver_stocks.select(
    "ticker",
    "date",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "daily_change_pct",
    "price_volatility"
).withColumn(
    "close_ma_7d", F.round(F.avg("close").over(window_7day), 2)
).withColumn(
    "close_ma_30d", F.round(F.avg("close").over(window_30day), 2)
).withColumn(
    "close_ma_90d", F.round(F.avg("close").over(window_90day), 2)
).withColumn(
    "volume_ma_7d", F.round(F.avg("volume").over(window_7day), 0)
).withColumn(
    "volume_ma_30d", F.round(F.avg("volume").over(window_30day), 0)
).withColumn(
    "volatility_ma_7d", F.round(F.avg("price_volatility").over(window_7day), 2)
).withColumn(
    "volatility_ma_30d", F.round(F.avg("price_volatility").over(window_30day), 2)
)

# Add momentum indicators
rolling_avg_df = rolling_avg_df \
    .withColumn("price_vs_ma_7d", 
                F.round((F.col("close") - F.col("close_ma_7d")) / F.col("close_ma_7d") * 100, 2)) \
    .withColumn("price_vs_ma_30d", 
                F.round((F.col("close") - F.col("close_ma_30d")) / F.col("close_ma_30d") * 100, 2)) \
    .withColumn("ma_7d_vs_ma_30d", 
                F.round((F.col("close_ma_7d") - F.col("close_ma_30d")) / F.col("close_ma_30d") * 100, 2))

print(f"Rolling averages calculated for {rolling_avg_df.count():,} records")
print("\n📋 Sample Rolling Averages (Recent Data):")
display(rolling_avg_df.orderBy(F.col("date").desc()).limit(10))

In [0]:
# ============================
# SAVE TO GOLD LAYER
# ============================

print("📦 Saving aggregated data to gold layer...\n")

# Create gold schema if not exists
spark.sql("CREATE SCHEMA IF NOT EXISTS financial_market.gold")

# 1. Save daily aggregations
print("1. Saving daily market aggregations...")
daily_agg.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("financial_market.gold.stocks_daily_summary")
print("   ✅ Saved to financial_market.gold.stocks_daily_summary")

# 2. Save ticker aggregations
print("\n2. Saving ticker aggregations...")
ticker_agg.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("financial_market.gold.stocks_ticker_summary")
print("   ✅ Saved to financial_market.gold.stocks_ticker_summary")

# 3. Save rolling averages
print("\n3. Saving rolling averages and trends...")
rolling_avg_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("financial_market.gold.stocks_trends")
print("   ✅ Saved to financial_market.gold.stocks_trends")

print("\n✅ All gold layer tables created successfully!")
print("\n📊 Gold Layer Tables:")
print("  - financial_market.gold.stocks_daily_summary")
print("  - financial_market.gold.stocks_ticker_summary")
print("  - financial_market.gold.stocks_trends")

In [0]:
# ============================
# KEY INSIGHTS & TRADING SIGNALS
# ============================

print("🔍 Generating key insights from rolling averages...\n")

# Get latest data for each ticker
latest_window = Window.partitionBy("ticker").orderBy(F.col("date").desc())
latest_trends = rolling_avg_df \
    .withColumn("row_num", F.row_number().over(latest_window)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

# 1. Stocks trading above their moving averages (Bullish signal)
print("1️⃣ BULLISH SIGNALS (Price > 30-day MA):")
bullish = latest_trends.filter(
    (F.col("price_vs_ma_30d") > 0) & 
    (F.col("ma_7d_vs_ma_30d") > 0)
).select(
    "ticker", "date", "close", "close_ma_7d", "close_ma_30d",
    "price_vs_ma_30d", "ma_7d_vs_ma_30d"
).orderBy(F.col("price_vs_ma_30d").desc())

print(f"   Found {bullish.count()} bullish stocks")
display(bullish.limit(5))

# 2. Stocks trading below their moving averages (Bearish signal)
print("\n2️⃣ BEARISH SIGNALS (Price < 30-day MA):")
bearish = latest_trends.filter(
    (F.col("price_vs_ma_30d") < 0) & 
    (F.col("ma_7d_vs_ma_30d") < 0)
).select(
    "ticker", "date", "close", "close_ma_7d", "close_ma_30d",
    "price_vs_ma_30d", "ma_7d_vs_ma_30d"
).orderBy(F.col("price_vs_ma_30d").asc())

print(f"   Found {bearish.count()} bearish stocks")
display(bearish.limit(5))

# 3. High volume stocks
print("\n3️⃣ HIGH VOLUME STOCKS (Volume > 30-day MA):")
high_volume = latest_trends.filter(
    F.col("volume") > F.col("volume_ma_30d") * 1.5
).select(
    "ticker", "date", "volume", "volume_ma_30d",
    F.round((F.col("volume") / F.col("volume_ma_30d") - 1) * 100, 2).alias("volume_spike_pct")
).orderBy(F.col("volume_spike_pct").desc())

print(f"   Found {high_volume.count()} high volume stocks")
display(high_volume.limit(5))

print("\n✅ Analysis complete!")